%matplotlib notebook

In [ ]:
import numpy as np 
import matplotlib.pyplot as plt 
from math import ceil, log; from typing import Any, Callable, Mapping, cast


def plot_3dsphere(ax: Any, centre_pt: np.ndarray, radius: float) -> None :  
    if centre_pt.shape != (3, 1):
        raise ValueError("Center must be a numpy array of shape (3, 1)")

    # Create spherical coordinate grid
    u = np.linspace(0, 2 * np.pi, 100)
    v = np.linspace(0, np.pi, 100)

    # Parametric equations for sphere
    x = radius * np.outer(np.cos(u), np.sin(v)) + centre_pt[0, 0]
    y = radius * np.outer(np.sin(u), np.sin(v)) + centre_pt[1, 0]
    z = radius * np.outer(np.ones_like(u), np.cos(v)) + centre_pt[2, 0]

    # Plot on provided axis
    ax.plot_surface(x, y, z, color='r', alpha=0.6, edgecolor='k')

    # Optionally mark the center
    ax.scatter(centre_pt[0, 0], centre_pt[1, 0], centre_pt[2, 0], color='r', s=50) 


def plot_2dsphere(ax: Any, centre_pt: np.ndarray, radius: float) -> None : 
    if centre_pt.shape != (2, 1):
        raise ValueError("Center must be a numpy array of shape (2, 1)")

    circle = plt.Circle(
        (centre_pt[0, 0], centre_pt[1, 0]),
        radius,
        color= 'b',
        alpha= 0.6,
        ec= 'k',
        lw= 1.5
    )
    ax.add_patch(circle)

    # Optionally mark the center
    ax.scatter(centre_pt[0, 0], centre_pt[1, 0], color='b', s=30) 


def plot_sphere(ax, center: np.ndarray, radius: float, color='b', alpha=0.4, edgecolor='k'):
    if center.shape != (3, 1):
        raise ValueError("Center must be a numpy array of shape (3, 1)")

    u = np.linspace(0, 2 * np.pi, 100)
    v = np.linspace(0, np.pi, 100)

    x = radius * np.outer(np.cos(u), np.sin(v)) + center[0, 0]
    y = radius * np.outer(np.sin(u), np.sin(v)) + center[1, 0]
    z = radius * np.outer(np.ones_like(u), np.cos(v)) + center[2, 0]

    ax.plot_surface(x, y, z, color=color, alpha=alpha, edgecolor=edgecolor)
    ax.scatter(center[0, 0], center[1, 0], center[2, 0], color=color, s=50)

def plot_circle_on_plane(ax, center: np.ndarray, radius: float, z_level=0.0, color='r', alpha=0.7):
    if center.shape != (2, 1):
        raise ValueError("Center must be a numpy array of shape (2, 1)")

    theta = np.linspace(0, 2*np.pi, 200)
    x = radius * np.cos(theta) + center[0, 0]
    y = radius * np.sin(theta) + center[1, 0]
    z = np.ones_like(x) * z_level  # fixed z-plane

    ax.plot(x, y, z, color=color, alpha=alpha, linewidth=2)
    ax.scatter(center[0, 0], center[1, 0], z_level, color=color, s=30)

def get_random_pt(centre_pt: np.ndarray, delta: float) -> np.ndarray : 
    if centre_pt.shape != (3, 1):
        raise ValueError("Center must be a numpy array of shape (3, 1)")

    # Step 1: Random direction (uniform on unit sphere)
    direction = np.random.normal(size=(3, 1))
    direction /= np.linalg.norm(direction)

    # Step 2: Random radius with cubic distribution for uniformity in volume
    r = delta * np.random.rand() ** (1/3)

    # Step 3: Combine
    point = centre_pt + r * direction

    return point

def construct_covariance_matrix(obj_fn_info: Mapping[str, Any], centre_pt: np.ndarray, delta: float) -> np.ndarray : 
    no_solns = ceil(20 * log(2))
    gradient = cast(Callable[[np.ndarray], np.ndarray], obj_fn_info['gradient']) 

    #Get gradients of objective function at no_solns-1 random places in the ball  
    gradients = [gradient(centre_pt)]  
    for _ in range(no_solns-1) : 
        random_pt = get_random_pt(centre_pt, delta)  
        grad_at_pt = gradient(random_pt) 
        gradients.append(grad_at_pt) 


    #should have shape (3,3)
    C = sum([np.outer(a,a) for a in gradients])/no_solns

    return C


def construct_active_subspace(obj_fn_info: Mapping[str, Any], centre_pt: np.ndarray, delta: float) -> np.ndarray : 
    d = 2 #subspace dim
    covar_matrix = construct_covariance_matrix(obj_fn_info, centre_pt, delta)

    #find eigenpairs
    eigenvalues, eigenvectors = np.linalg.eigh(covar_matrix)
    
    # Create list of (eigenvalue, eigenvector) tuples
    eigenpairs = [(eigenvalues[i], eigenvectors[:, i]) for i in range(len(eigenvalues))]
    
    # Sort by eigenvalue in descending order
    eigenpairs.sort(key=lambda pair: pair[0], reverse=True)

    active_subspace_eigenvects = [a[1].reshape(-1,1) for a in eigenpairs[:2]]   




    active_subspace = np.hstack(active_subspace_eigenvects)  

    if active_subspace.shape != (3,2) : 
        raise ValueError(
            "Active subspace has the wrong dimensions"
            )     

    return active_subspace

def plot_3d_function(ax, obj_fn_info, domain_limits, num_levels=5):
    """
    Plots isosurfaces of a 3D function f(x) = ||x||^2_2 over a cubic domain.
    For ||x||^2, these are concentric spheres of increasing radius.
    """
    f = obj_fn_info["function"]

    # Create a grid in 3D
    x = np.linspace(domain_limits[0], domain_limits[1], 50)
    y = np.linspace(domain_limits[0], domain_limits[1], 50)
    z = np.linspace(domain_limits[0], domain_limits[1], 50)
    X, Y, Z = np.meshgrid(x, y, z)

    # Evaluate the function on the grid
    F = X**2 + Y**2 + Z**2  # since f(x)=||x||^2

    # Choose isosurface levels
    levels = np.linspace(F.min(), F.max(), num_levels + 2)[1:-1]

    # Plot each isosurface as a translucent sphere
    for val in levels:
        r = np.sqrt(val)
        u = np.linspace(0, 2 * np.pi, 50)
        v = np.linspace(0, np.pi, 50)
        xs = r * np.outer(np.cos(u), np.sin(v))
        ys = r * np.outer(np.sin(u), np.sin(v))
        zs = r * np.outer(np.ones_like(u), np.cos(v))
        ax.plot_surface(xs, ys, zs, color=plt.get_cmap('viridis')(val / F.max()), alpha=0.3, linewidth=0)


def plot_response_surface(ax, obj_fn_info, active_subspace, proj_centre, delta=3.0, z_offset=-5.0):
    """
    Plots the response surface f(x) = ||x||^2 over the 2D active subspace coordinates (y1, y2),
    mapped back into 3D for visualization.
    """
    f = obj_fn_info["function"]

    # Grid in 2D active subspace coordinates
    y1 = np.linspace(proj_centre[0, 0] - delta, proj_centre[0, 0] + delta, 50)
    y2 = np.linspace(proj_centre[1, 0] - delta, proj_centre[1, 0] + delta, 50)
    Y1, Y2 = np.meshgrid(y1, y2)

    # Map each (y1, y2) point back to 3D space via active subspace
    points2d = np.vstack([Y1.flatten(), Y2.flatten()])
    points3d = active_subspace @ points2d

    # Evaluate f(x) at each point
    values = np.array([f(points3d[:, i].reshape(-1, 1)) for i in range(points3d.shape[1])])
    F = values.reshape(Y1.shape)

    # Shift the surface downward for visual clarity
    Xs, Ys, Zs = (
        points3d[0, :].reshape(Y1.shape),
        points3d[1, :].reshape(Y2.shape),
        points3d[2, :].reshape(Y1.shape) + z_offset
    )

    # Plot the response surface
    surf = ax.plot_surface(
        Xs, Ys, Zs, facecolors=plt.get_cmap('viridis')((F - F.min()) / (F.max() - F.min())),
        rstride=1, cstride=1, antialiased=True, alpha=0.8
    )

    # Add a colorbar
    mappable = plt.cm.ScalarMappable(cmap="viridis")
    mappable.set_array(F)
    plt.colorbar(mappable, ax=ax, shrink=0.5, pad=0.1, label="f(x) = ||x||²")


def main():
    obj_fn_info = {
        "function": lambda x: np.sum(x**2),
        
        "gradient": lambda x: 2 * x
    }

    orig_centre = np.array([[10], [3], [2]])
    radius = 2.5

    # --- Construct active subspace ---
    active_subspace = construct_active_subspace(obj_fn_info, orig_centre, radius)
    proj_centre = active_subspace.T @ orig_centre  # 2D projection

    # --- Plot setup ---
    fig = plt.figure(figsize=(10, 8))
    ax3d = fig.add_subplot(111, projection='3d')


    # --- Plot the 3D function (isosurfaces) ---
    # plot_3d_function(ax3d, obj_fn_info, domain_limits=(-12, 12), num_levels=6)

    # --- Plot the 3D sphere ---
    plot_3dsphere(ax3d, orig_centre, radius)

    # --- Sample points on sphere surface ---
    u = np.linspace(0, 2 * np.pi, 80)
    v = np.linspace(0, np.pi, 80)
    x = radius * np.outer(np.cos(u), np.sin(v)) + orig_centre[0, 0]
    y = radius * np.outer(np.sin(u), np.sin(v)) + orig_centre[1, 0]
    z = radius * np.outer(np.ones_like(u), np.cos(v)) + orig_centre[2, 0]

    points3d = np.vstack([x.flatten(), y.flatten(), z.flatten()])
    projected_points = active_subspace.T @ points3d  # (2, N)

    # --- Map 2D projection back into 3D space ---
    # We'll "lift" it back to 3D using the active subspace basis
    # This gives us a plane inside 3D aligned with the subspace
    points3d_projected_plane = active_subspace @ projected_points

    # Shift the projected shadow below the original sphere
    # so it doesn’t overlap visually (just for clarity)
    offset = -3.0
    points3d_projected_plane_shifted = points3d_projected_plane + np.array([[0], [0], [offset]])

    # --- Plot the projected “shadow” ---
    ax3d.scatter(
        points3d_projected_plane_shifted[0, :],
        points3d_projected_plane_shifted[1, :],
        points3d_projected_plane_shifted[2, :],
        color='b', alpha=0.4, s=8, label="Projected (2D) ball"
    )

    # --- Plot the response surface under the sphere ---
    plot_response_surface(ax3d, obj_fn_info, active_subspace, proj_centre, delta=5.0, z_offset=-5.0)


    # --- Compute and plot projected center in 3D ---
    projected_center_3d = active_subspace @ proj_centre + np.array([[0], [0], [offset]])

    # --- Plot centers ---
    ax3d.scatter(orig_centre[0, 0], orig_centre[1, 0], orig_centre[2, 0],
                 color='r', s=80, label='Original centre', edgecolor='k')
    ax3d.scatter(projected_center_3d[0, 0], projected_center_3d[1, 0], projected_center_3d[2, 0],
                 color='b', s=80, label='Projected centre', edgecolor='k')

    # --- Connect centers with a dashed line ---
    ax3d.plot(
        [orig_centre[0, 0], projected_center_3d[0, 0]],
        [orig_centre[1, 0], projected_center_3d[1, 0]],
        [orig_centre[2, 0], projected_center_3d[2, 0]],
        color='k', linestyle='--', linewidth=2, label="Projection mapping"
    )

    # --- Add annotations for clarity ---
    ax3d.text(orig_centre[0, 0], orig_centre[1, 0], orig_centre[2, 0] + 0.5,
              "orig_centre", color='r', fontsize=10, weight='bold')
    ax3d.text(projected_center_3d[0, 0], projected_center_3d[1, 0],
              projected_center_3d[2, 0] - 0.5, "proj_centre", color='b', fontsize=10, weight='bold')


    for angle in range(0, 360, 2):
        ax3d.view_init(elev=30, azim=angle)
        plt.draw()
        plt.pause(0.05)

    # --- Label and format ---
    ax3d.set_title("3D Sphere with Overlaid 2D Active Subspace Projection")
    ax3d.set_xlabel("X-axis")
    ax3d.set_ylabel("Y-axis")
    ax3d.set_zlabel("Z-axis")
    ax3d.legend()
    ax3d.set_box_aspect([1, 1, 1])
    plt.tight_layout()
    plt.show()

if __name__ == '__main__' : 
    main() 